# Benchmark Completo — Transfer Learning no CIFAR-10 (versão atual)
**Subset 5.000 imagens · 10 épocas · GPU**

Mesma configuração dos labs: na Feature Extraction e no Fine-Tuning Parcial o backbone fica **100% congelado** (inclusive as estatísticas da BatchNorm), a `fc` do Fine-Tuning Parcial usa `lr=1e-2`, e não há Data Augmentation. Roda 4 experimentos em sequência e gera uma tabela comparativa final.

| # | Estratégia | Params treináveis |
|---|---|---|
| 1 | Scratch (do zero) | 11.2M |
| 2 | Feature Extraction | 5.1K |
| 3 | Fine-Tuning Parcial (layer4) | 8.4M |
| 4 | Fine-Tuning Completo | 11.2M |

> Os resultados e tempos salvos neste notebook foram gerados numa GPU local (NVIDIA RTX 5050), não numa T4 do Colab: a acurácia é comparável, os tempos não.

## ▶️ Passo 0 — Ativar a GPU

**Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**

Execute a célula abaixo para confirmar.

In [1]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'✅ GPU ativa: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('⚠️  GPU NÃO detectada — ative em: Ambiente de execução → Alterar o tipo → GPU (T4)')
    print('   Sem GPU os experimentos levam várias horas.')

✅ GPU ativa: NVIDIA GeForce RTX 5050 Laptop GPU


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Subset, DataLoader
import numpy as np
import time

torch.manual_seed(42)

NUM_EPOCHS = 10
N_TRAIN    = 5000
N_VAL      = 1000
BATCH      = 64

results = {}   # acumula resultados de todos os experimentos
print(f'Config: {N_TRAIN} treino | {N_VAL} val | {NUM_EPOCHS} épocas | batch={BATCH} | device={device}')

Config: 5000 treino | 1000 val | 10 épocas | batch=64 | device=cuda


In [3]:
# ── Transforms ─────────────────────────────────────────────────────────────
transform_std = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_balanced_subset(dataset, n_total):
    n_per_class = n_total // 10
    indices, counts = [], {c: 0 for c in range(10)}
    for idx, label in enumerate(dataset.targets):
        if counts[label] < n_per_class:
            indices.append(idx)
            counts[label] += 1
        if len(indices) == n_total:
            break
    return Subset(dataset, indices)

# ── Download rápido do CIFAR-10 ────────────────────────────────────────────
# O servidor oficial (Univ. de Toronto) está muito lento. Baixamos o MESMO arquivo de
# um espelho rápido; o torchvision confere o MD5 e, se não bater (ou se o espelho
# estiver fora do ar), baixa do servidor oficial automaticamente.
import os, urllib.request
if not os.path.exists('data/cifar-10-batches-py'):
    os.makedirs('data', exist_ok=True)
    try:
        urllib.request.urlretrieve(
            'https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz',
            'data/cifar-10-python.tar.gz')
    except Exception as e:
        print(f'Espelho indisponível ({e}); usando o servidor oficial.')

# ── Datasets padrão (usados na maioria dos experimentos) ───────────────────
train_full_std = datasets.CIFAR10(root='data', train=True,  download=True, transform=transform_std)
val_full       = datasets.CIFAR10(root='data', train=False, download=True, transform=transform_std)

train_ds_std = extract_balanced_subset(train_full_std, N_TRAIN)
val_ds       = extract_balanced_subset(val_full, N_VAL)

train_loader_std = DataLoader(train_ds_std, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader       = DataLoader(val_ds,       batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Dados carregados: {N_TRAIN} treino | {N_VAL} val')

✅ Dados carregados: 5000 treino | 1000 val


In [4]:
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, freeze_bn=False):
    model.train()
    if freeze_bn:   # backbone pré-treinado 100% congelado (estatísticas da BatchNorm do ImageNet)
        for mod in model.modules():
            if isinstance(mod, nn.BatchNorm2d):
                mod.eval()
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, preds = torch.max(model(inputs), 1)
            correct += (preds == labels).sum().item()
    return correct / len(loader.dataset)

def run_experiment(name, model, optimizer, train_loader, n_epochs=NUM_EPOCHS, freeze_bn=False):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'  Parâmetros treináveis: {trainable:,}')
    print(f'{"="*55}')
    t0 = time.time()
    acc = 0.0
    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, freeze_bn)
        acc  = evaluate(model, val_loader)
        print(f'  Época {epoch:>2}/{n_epochs} | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}%')
    elapsed = time.time() - t0
    print(f'  ✅ {elapsed/60:.1f} min | Acurácia final: {acc*100:.2f}%')
    results[name] = {'acc': acc * 100, 'time_min': elapsed / 60, 'params': trainable}
    return model

print('✅ Funções auxiliares definidas.')

✅ Funções auxiliares definidas.


## Experimento 1 — Scratch (do zero)
ResNet-18 com pesos **aleatórios**, sem pré-treinamento. Linha de base inferior: mostra o custo de não usar Transfer Learning.

In [5]:
torch.manual_seed(42)
m = models.resnet18(weights=None)
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
_ = run_experiment('Scratch (do zero)', m, opt, train_loader_std)


  Scratch (do zero)
  Parâmetros treináveis: 11,181,642


  Época  1/10 | Loss: 2.0138 | Val Acc: 29.80%


  Época  2/10 | Loss: 1.6925 | Val Acc: 30.40%


  Época  3/10 | Loss: 1.4635 | Val Acc: 37.80%


  Época  4/10 | Loss: 1.3300 | Val Acc: 38.60%


...

  Época  6/10 | Loss: 0.9468 | Val Acc: 44.90%


  Época  7/10 | Loss: 0.7116 | Val Acc: 37.80%


  Época  8/10 | Loss: 0.5480 | Val Acc: 42.40%


  Época  9/10 | Loss: 0.3934 | Val Acc: 36.00%


  Época 10/10 | Loss: 0.2754 | Val Acc: 45.00%
  ✅ 3.0 min | Acurácia final: 45.00%


## Experimento 2 — Feature Extraction
Backbone **100% congelado** (pesos e BatchNorm do ImageNet); treina apenas a camada `fc` (5.130 parâmetros). Estratégia do Lab 1.

In [6]:
torch.manual_seed(42)
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in m.parameters():
    p.requires_grad = False
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.fc.parameters(), lr=0.01, momentum=0.9)
_ = run_experiment('Feature Extraction', m, opt, train_loader_std, freeze_bn=True)


  Feature Extraction
  Parâmetros treináveis: 5,130


  Época  1/10 | Loss: 0.8605 | Val Acc: 79.60%


  Época  2/10 | Loss: 0.4455 | Val Acc: 83.80%


  Época  3/10 | Loss: 0.3933 | Val Acc: 84.80%


  Época  4/10 | Loss: 0.3582 | Val Acc: 82.50%


...

  Época  6/10 | Loss: 0.3259 | Val Acc: 85.40%


  Época  7/10 | Loss: 0.2961 | Val Acc: 84.60%


  Época  8/10 | Loss: 0.2806 | Val Acc: 84.40%


  Época  9/10 | Loss: 0.2740 | Val Acc: 85.30%


  Época 10/10 | Loss: 0.2629 | Val Acc: 84.80%
  ✅ 2.0 min | Acurácia final: 84.80%


## Experimento 3 — Fine-Tuning Parcial (layer4)
Descongela apenas o último bloco residual (`layer4`, `lr=1e-4`) e treina a nova `fc` com `lr=1e-2`; o restante, inclusive a BatchNorm, fica congelado. Estratégia do Lab 2.

In [7]:
torch.manual_seed(42)
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in m.parameters():
    p.requires_grad = False
for p in m.layer4.parameters():
    p.requires_grad = True
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(
    [{'params': m.layer4.parameters(), 'lr': 1e-4},
     {'params': m.fc.parameters(),    'lr': 1e-2}],
    momentum=0.9)
_ = run_experiment('Fine-Tuning Parcial (layer4)', m, opt, train_loader_std, freeze_bn=True)


  Fine-Tuning Parcial (layer4)
  Parâmetros treináveis: 8,398,858


  Época  1/10 | Loss: 0.8265 | Val Acc: 82.30%


  Época  2/10 | Loss: 0.3858 | Val Acc: 85.50%


  Época  3/10 | Loss: 0.3102 | Val Acc: 87.30%


  Época  4/10 | Loss: 0.2638 | Val Acc: 85.00%


...

  Época  6/10 | Loss: 0.1949 | Val Acc: 85.80%


  Época  7/10 | Loss: 0.1577 | Val Acc: 87.10%


  Época  8/10 | Loss: 0.1368 | Val Acc: 87.40%


  Época  9/10 | Loss: 0.1109 | Val Acc: 87.10%


  Época 10/10 | Loss: 0.0984 | Val Acc: 88.00%
  ✅ 2.3 min | Acurácia final: 88.00%


## Experimento 4 — Fine-Tuning Completo
Todas as camadas abertas para atualização (inclusive a BatchNorm), com `lr=0.001` para evitar *catastrophic forgetting*. Teto de desempenho com este dataset.

In [8]:
torch.manual_seed(42)
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.parameters(), lr=0.001, momentum=0.9)
_ = run_experiment('Fine-Tuning Completo', m, opt, train_loader_std)


  Fine-Tuning Completo
  Parâmetros treináveis: 11,181,642


  Época  1/10 | Loss: 1.5776 | Val Acc: 73.20%


  Época  2/10 | Loss: 0.6578 | Val Acc: 84.10%


  Época  3/10 | Loss: 0.3994 | Val Acc: 86.60%


  Época  4/10 | Loss: 0.2477 | Val Acc: 87.20%


...

  Época  6/10 | Loss: 0.1148 | Val Acc: 88.60%


  Época  7/10 | Loss: 0.0852 | Val Acc: 87.90%


  Época  8/10 | Loss: 0.0700 | Val Acc: 89.00%


  Época  9/10 | Loss: 0.0425 | Val Acc: 88.80%


  Época 10/10 | Loss: 0.0341 | Val Acc: 87.70%
  ✅ 3.0 min | Acurácia final: 87.70%


## Tabela de Resultados

In [9]:
print(f'\n{"="*65}')
print(f'{"BENCHMARK COMPLETO — CIFAR-10 · 5.000 imagens · 10 épocas":^65}')
print(f'{"="*65}')
print(f'{"Experimento":<30} {"Params":>12} {"Acurácia":>10} {"Tempo":>10}')
print(f'{"-"*65}')
for name, r in results.items():
    print(f'{name:<30} {r["params"]:>12,} {r["acc"]:>9.2f}% {r["time_min"]:>8.1f}min')
print(f'{"="*65}')
best = max(results, key=lambda k: results[k]['acc'])
print(f'  Melhor: {best} → {results[best]["acc"]:.2f}%')


    BENCHMARK COMPLETO — CIFAR-10 · 5.000 imagens · 10 épocas    
Experimento                          Params   Acurácia      Tempo
-----------------------------------------------------------------
Scratch (do zero)                11,181,642     45.00%      3.0min
Feature Extraction                    5,130     84.80%      2.0min
Fine-Tuning Parcial (layer4)      8,398,858     88.00%      2.3min
Fine-Tuning Completo             11,181,642     87.70%      3.0min
  Melhor: Fine-Tuning Parcial (layer4) → 88.00%
